In [1]:
import polars as pl
from fontTools.misc.timeTools import epoch_diff
from soupsieve import select
import os
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from gensim.models import Word2Vec
import numpy as np
from cuml.neighbors import NearestNeighbors

In [2]:
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
transaction_path='../data/transactions.parquet'

In [3]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [4]:
sentences=(
    transactions.sort(['customer_id','time'])
    .group_by('customer_id')
    .agg(pl.col('article_id').tail(30))
    ['article_id']
    .to_list()
)

In [5]:
sentences[:10]

[[664074001, 664074013, 663942009, 243937001, 664074001, 771733004],
 [668815002,
  473954008,
  568601008,
  321249010,
  612935009,
  708855005,
  473954005,
  562245018,
  778534001,
  751557002,
  825318008,
  868879003,
  859136004,
  887593002],
 [710554001,
  680264001,
  760975001,
  721997001,
  663826001,
  722991001,
  723151003,
  883015001,
  893424003,
  644797009],
 [628621001, 656161001],
 [755360001, 755360001],
 [722376001,
  760223002,
  736592001,
  736593002,
  758118001,
  762977001,
  758060001,
  758064001,
  758125001,
  758064001,
  758110001,
  758002004,
  758156001,
  758157001,
  762977001,
  758808001,
  156231001,
  779889001,
  768465003,
  751471019,
  865314011,
  832453005,
  865314011,
  818403001,
  780297015,
  811500001,
  913341001,
  913341001,
  913340001,
  913340001],
 [863665002,
  772234001,
  870654002,
  870654001,
  830016001,
  831172001,
  870654002,
  870654001,
  830016001,
  779414001,
  708311006,
  827968001,
  751471001,
  72815

In [6]:
model=Word2Vec(
    sentences=sentences,
    vector_size=64,
    window=5,
    min_count=5,
    workers=5,
    sg=1,
    epochs=5,
)

In [7]:
model.save('../save/model/item2vec.model')

In [8]:
model=Word2Vec.load('../save/model/item2vec.model')

In [9]:
idx2aid=model.wv.index_to_key
idx2aid

[706016001,
 706016002,
 372860001,
 610776002,
 759871002,
 372860002,
 464297007,
 720125001,
 610776001,
 751471001,
 706016003,
 399256001,
 568601006,
 156231001,
 351484002,
 562245046,
 688537004,
 399223001,
 448509014,
 673396002,
 673677002,
 608776002,
 160442007,
 684209004,
 579541001,
 590928001,
 783346001,
 741356002,
 158340001,
 160442010,
 599580017,
 706016015,
 562245001,
 507909001,
 717490008,
 841383002,
 572797001,
 688537011,
 636323001,
 599580038,
 706016006,
 749699002,
 562245018,
 399256005,
 684209013,
 111586001,
 554450001,
 723469001,
 759871001,
 685814001,
 599580055,
 573716012,
 719655001,
 228257001,
 749699001,
 778064003,
 778064001,
 711053003,
 572797002,
 678942001,
 759465001,
 695632002,
 806388002,
 111593001,
 768912001,
 787946002,
 599580052,
 850917001,
 685816002,
 484398001,
 554598001,
 803757001,
 806388001,
 600886001,
 733749001,
 736530007,
 753737001,
 554479001,
 685816001,
 699080001,
 841383003,
 748355003,
 562245050,
 568

In [10]:
embeddings = np.array([model.wv[aid] for aid in model.wv.index_to_key], dtype=np.float32)# 词的向量
d=model.wv.vectors.shape[1]

In [11]:
knn=NearestNeighbors(n_neighbors=51,metric='euclidean') # 最近邻查找，每个样本返回21个（包括自己）
knn.fit(embeddings)

NearestNeighbors()

In [12]:
_,aid_nns=knn.kneighbors(embeddings)

In [13]:
aid_nns=aid_nns[:,1:]
aid_nns

array([[   40,     1,    10, ..., 64003, 71343, 76024],
       [   40,    31,     0, ..., 69363, 64465, 80947],
       [    5,   130,    21, ..., 65808, 81112,   208],
       ...,
       [81645, 83369, 81242, ..., 82488, 82605, 81562],
       [81908, 81422, 82023, ..., 75440, 82051, 79536],
       [81158, 82683, 83141, ..., 82452, 83294, 80716]])

In [14]:
item2item={idx2aid[aid]:[idx2aid[i] for i in row] for aid,row in enumerate(aid_nns)}
item2item

{706016001: [706016006,
  706016002,
  706016003,
  706016015,
  706016019,
  706016007,
  706016004,
  573085028,
  706016028,
  554450001,
  621381014,
  706016025,
  706016029,
  706016038,
  706016011,
  798579002,
  621381012,
  621381001,
  554450036,
  755754001,
  621381016,
  573085043,
  739823006,
  706016020,
  708679002,
  706016016,
  765739001,
  562820002,
  905776004,
  755754002,
  664133001,
  669999003,
  673901011,
  910439002,
  719530003,
  399223001,
  458083007,
  747507004,
  573085004,
  673901012,
  754039004,
  935003001,
  769074005,
  554450051,
  845626001,
  706016034,
  516859008,
  696500001,
  757382003,
  818978001],
 706016002: [706016006,
  706016015,
  706016001,
  706016003,
  706016004,
  706016007,
  706016019,
  706016029,
  621381014,
  708679002,
  621381001,
  706016011,
  706016016,
  706016038,
  573085004,
  706016020,
  706016028,
  573085028,
  554450036,
  554450004,
  673901012,
  621381016,
  554450001,
  573085043,
  673901011,
  

In [15]:
def train_w2vec(data,is_valid=False,model_path='../save/model/item2vec.model'):
    sentences=(
        data.sort(['customer_id','time'])
        .group_by('customer_id', maintain_order=True)
        .agg(pl.col('article_id').tail(30))
        ['article_id']
        .to_list()
    )
    if os.path.exists(model_path) and not is_valid:model=Word2Vec.load(model_path)
    else:
        model=Word2Vec(
            sentences=sentences,
            vector_size=64,
            window=5,
            min_count=5,
            workers=5,
            sg=1,
            epochs=5,
        )
    if not is_valid:model.save('../save/model/item2vec.model')

    return model

In [16]:
def recall_item(model,topk):
    embeddings = np.array([model.wv[aid] for aid in model.wv.index_to_key], dtype=np.float32)# 词的向量
    knn=NearestNeighbors(n_neighbors=topk+1,metric='euclidean') # 最近邻查找，每个样本返回21个（包括自己）
    knn.fit(embeddings)

    _,aid_nns=knn.kneighbors(embeddings)
    aid_nns=aid_nns[:,1:]

    idx2aid=model.wv.index_to_key

    item2item={idx2aid[aid]:[idx2aid[i] for i in row] for aid,row in enumerate(aid_nns)}
    return item2item

In [17]:
def recall_w2vec(data,topk=50,hist_len=10,is_valid=False,model_path='../save/model/item2vec.model'):
    rows=[]
    model=train_w2vec(data,is_valid,model_path)
    item2item=recall_item(model,topk)

    customer_ids=[]
    article_aids=[]
    ranks=[]

    for cid,g in tqdm(data.group_by('customer_id'),total=data['customer_id'].unique().shape[0]):
        hist=(
            g.sort('time',descending=True)
            .select('article_id')
            .head(hist_len)['article_id']
            .to_list()
        )
        score=defaultdict(float)

        for i,aid in enumerate(hist):
            if aid not in item2item:
                continue
            w=1.0/(i+1)

            for j ,rec_aid in enumerate(item2item[aid]):
                if rec_aid not in hist:
                   score[rec_aid]+=w/(j+1)
        res=sorted(score.items(),key=lambda x:x[1],reverse=True)[:topk]



        for rank, (aid,_) in enumerate(res):
            customer_ids.append(cid[0])
            article_aids.append(aid)
            ranks.append(rank)
    df = pl.DataFrame({
        "customer_id": customer_ids,
        "article_id": article_aids,
        "rank": ranks,
        }).with_columns(
            pl.col("rank").cast(pl.UInt8)
        )
    print('ok')
    return df

In [18]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [19]:
def metric_recall(data,topk=5):
    train_df,valid_df=get_validation_data(data)

    user_item=recall_w2vec(train_df,topk*10,5,True)
    user_set=set(user_item['customer_id'])

    pred_df = (
        user_item
        .sort("rank")
        .group_by("customer_id")
        .agg(pl.col("article_id").alias("pred_items"))
    )

    true_df = (
        valid_df
    .group_by("customer_id")
        .agg(pl.col('article_id').unique().alias("true_items")))

    eval_df=pred_df.join(true_df,on='customer_id',how='inner')

    for k in range(10, topk * 10 + 1, 10):
        total = 0.0
        cnt = 0

        for pred, true in zip(eval_df["pred_items"], eval_df["true_items"]):
            if len(true) == 0:
                continue
            total += len(set(pred[:k]) & set(true)) / len(true)
            cnt += 1

        print(f"Recall@{k}: {total / cnt:.6f}")

In [ ]:
metric_recall(transactions)

 80%|████████  | 251170/312215 [00:57<00:14, 4149.68it/s]

In [ ]:
res=recall_w2vec(transactions)

In [ ]:
res.to_pandas().info()

In [ ]:
res.write_parquet('../save/candidate/w2vec.parquet')